# Maximum covariate effect by target–leader pair

This standalone notebook reads a covariate-screening Excel workbook and
identifies, for every **pair × covariate** combination, the fitted distribution
with the largest meaningful information-criterion improvement.

The requested matrix has:

- **Rows:** target–leader pairs
- **Columns:** covariates
- **Values:** maximum qualifying \(\Delta\mathrm{AIC}\) (or
  \(\Delta\mathrm{AICc}\) when that is the metric stored in the input workbook)
  across the fitted distributions

A result qualifies when:

1. the model is estimable;
2. the optimizer converged, when that field is available;
3. \(\Delta\mathrm{AIC}\geq 2\);
4. likelihood-ratio \(p<0.05\); and
5. GOF acceptance is preserved, when that field is available and the option
   below is enabled.

> **Important:** statistical significance is assessed using the likelihood-ratio
> p-value, not by checking whether \(\beta<0.05\). The coefficient \(\beta\)
> describes effect direction and size.


In [1]:
# ============================================================
# 1. USER CONFIGURATION
# ============================================================

from pathlib import Path
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
    HAS_IPYTHON_DISPLAY = True
except ImportError:
    HAS_IPYTHON_DISPLAY = False

    def display(value):
        if isinstance(value, pd.DataFrame):
            print(value.to_string())
        else:
            print(value)

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")


# Set this to an Excel path when you want to force a particular workbook.
# Leave as None to detect a compatible screening workbook automatically.
#
# Examples:
# INPUT_XLSX = Path("T4_41_Distribution_Covariate_Screening.xlsx")
# INPUT_XLSX = Path("Tables/T4_Covariate_Selection.xlsx")
INPUT_XLSX = r"D:\Headway\final run\tables\T4_41_Distribution_Covariate_Screening.xlsx"


# The summary workbook written by this notebook.
OUTPUT_XLSX = Path("T4_Maximum_Covariate_Effect_by_Pair.xlsx")


# Selection criteria
MIN_DELTA_AIC = 2.0
ALPHA_LR = 0.05

# If the input has a "GOF acceptance preserved" field, require it.
# If that field is absent, the notebook reports that GOF could not be applied.
REQUIRE_GOF_PRESERVED_IF_AVAILABLE = True


# Preferred manuscript ordering and labels
COVARIATE_ORDER = [
    "target_speed",
    "leading_speed",
    "speed_difference",
    "occupancy",
    "off_centeredness",
    "site",
    "flow",
]

COVARIATE_DISPLAY = {
    "target_speed": "Target Vehicle Speed",
    "leading_speed": "Leading Vehicle Speed",
    "speed_difference": "Speed Difference",
    "occupancy": "Occupancy",
    "off_centeredness": "Off-centeredness",
    "site": "Site",
    "flow": "Flow",
}


## Input compatibility

The loader recognizes both formats already used in this project:

- the newer 41-model workbook with columns such as `Distribution`, `ΔAIC`,
  `β`, `LR p`, and `GOF acceptance preserved`; and
- the earlier screening workbook with columns such as `dist_label`, `dAICc`,
  `beta`, and `LR_p`.

The workbook and compatible results sheet are selected by their contents rather
than by a fixed filename.


In [2]:
# ============================================================
# 2. HELPERS: DETECTION, NORMALIZATION, AND VALIDATION
# ============================================================

def normalized_token(value):
    '''Normalize a label so Unicode and punctuation variants can be matched.'''
    text = unicodedata.normalize("NFKC", str(value)).strip().casefold()
    replacements = {
        "δ": "delta",
        "β": "beta",
        "χ": "chi",
        "²": "2",
        "≥": "ge",
        "≤": "le",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return re.sub(r"[^a-z0-9]+", "", text)


COLUMN_ALIASES = {
    "Pair": [
        "Pair",
        "Stratum",
    ],
    "Distribution": [
        "Distribution",
        "dist_label",
        "Distribution label",
    ],
    "Covariate": [
        "Covariate",
        "Predictor",
    ],
    "Delta_AIC": [
        "ΔAIC",
        "Delta AIC",
        "dAIC",
        "ΔAICc",
        "Delta AICc",
        "dAICc",
    ],
    "Beta": [
        "β",
        "beta",
        "Coefficient",
    ],
    "LR_p": [
        "LR p",
        "LR_p",
        "LRT p",
        "Likelihood ratio p",
        "Likelihood-ratio p",
    ],
    "Estimable": [
        "Estimable",
        "estimable",
    ],
    "Optimizer_converged": [
        "Optimizer converged",
        "Converged",
        "optimizer_converged",
    ],
    "GOF_preserved": [
        "GOF acceptance preserved",
        "GOF preserved",
        "gof_preserved",
    ],
}


def detect_column_mapping(columns):
    '''Return standardized-name -> actual-column mapping.'''
    token_to_actual = {
        normalized_token(column): column
        for column in columns
    }
    mapping = {}
    for standard_name, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            token = normalized_token(alias)
            if token in token_to_actual:
                mapping[standard_name] = token_to_actual[token]
                break
    return mapping


def to_nullable_boolean(series):
    '''Convert Boolean, 0/1, and common text forms to nullable Boolean.'''
    if pd.api.types.is_bool_dtype(series):
        return series.astype("boolean")

    result = pd.Series(pd.NA, index=series.index, dtype="boolean")
    text = series.astype("string").str.strip().str.casefold()

    result.loc[text.isin(["true", "1", "yes", "y", "accepted", "pass"])] = True
    result.loc[text.isin(["false", "0", "no", "n", "rejected", "fail"])] = False
    return result


def canonical_covariate(value):
    '''Map project-specific covariate spellings to one stable key.'''
    token = normalized_token(value)

    aliases = {
        "target_speed": {
            "targetspeed",
            "targetvehiclespeed",
            "targetspeedkmhr",
        },
        "leading_speed": {
            "leaderspeed",
            "leadingspeed",
            "leadingvehiclespeed",
            "leadingspeedkmhr",
        },
        "speed_difference": {
            "speeddifference",
            "differentialspeed",
        },
        "occupancy": {
            "occupancy",
        },
        "off_centeredness": {
            "offcenteredness",
            "offcentredness",
            "offcentered",
            "offcentred",
        },
        "site": {
            "site",
            "location",
        },
        "flow": {
            "flow",
            "trafficflow",
            "flowpcuhrm",
            "flowpcuhm",
        },
    }

    for key, candidates in aliases.items():
        if token in candidates:
            return key

    # Preserve an unrecognized covariate rather than silently dropping it.
    return f"other__{token}"


def covariate_label(key, raw_value=None):
    if key in COVARIATE_DISPLAY:
        return COVARIATE_DISPLAY[key]
    if raw_value is not None:
        return str(raw_value)
    return key.replace("other__", "").replace("_", " ").title()


def pair_label(value):
    '''Convert machine pair labels into manuscript-friendly labels.'''
    text = str(value).strip()
    text = text.replace("_following_", " → ")
    text = text.replace(" following ", " → ")
    return text


def candidate_workbooks(explicit_path=None):
    '''List explicit or locally discoverable Excel workbooks.'''
    if explicit_path is not None:
        explicit = Path(explicit_path).expanduser()
        if not explicit.exists():
            raise FileNotFoundError(f"Input workbook not found: {explicit}")
        return [explicit.resolve()]

    roots = [
        Path.cwd(),
        Path.cwd() / "Tables",
        Path.cwd() / "tables",
        Path.cwd() / "upload",
        Path.cwd() / "project_sources",
    ]

    candidates = []
    seen = set()

    for root in roots:
        if not root.exists():
            continue

        iterator = root.glob("*.xlsx") if root == Path.cwd() else root.rglob("*.xlsx")
        for path in iterator:
            resolved = path.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)

            if path.name.startswith("~$"):
                continue
            if path.name == OUTPUT_XLSX.name:
                continue
            candidates.append(resolved)

    return candidates


def find_compatible_screening_sheet(explicit_path=None):
    '''
    Find the best workbook/sheet containing pairwise covariate-screening rows.
    '''
    required = {
        "Pair",
        "Distribution",
        "Covariate",
        "Delta_AIC",
        "Beta",
        "LR_p",
    }

    matches = []
    failures = []

    for workbook_path in candidate_workbooks(explicit_path):
        try:
            excel_file = pd.ExcelFile(workbook_path)
        except Exception as exc:
            failures.append(f"{workbook_path.name}: {exc}")
            continue

        for sheet_name in excel_file.sheet_names:
            try:
                preview = pd.read_excel(
                    workbook_path,
                    sheet_name=sheet_name,
                    nrows=5,
                )
            except Exception:
                continue

            mapping = detect_column_mapping(preview.columns)
            if not required.issubset(mapping):
                continue

            sheet_token = normalized_token(sheet_name)
            score = 100
            score += 35 if "fullscreen" in sheet_token else 0
            score += 20 if "GOF_preserved" in mapping else 0
            score += 15 if normalized_token(mapping["Delta_AIC"]) == "deltaaic" else 0
            score += 5 if "s10" in sheet_token else 0

            matches.append(
                {
                    "score": score,
                    "path": workbook_path,
                    "sheet": sheet_name,
                    "mapping": mapping,
                }
            )

    if not matches:
        searched = "\n".join(f"- {path}" for path in candidate_workbooks(explicit_path))
        raise FileNotFoundError(
            "No compatible covariate-screening sheet was found.\n"
            "A compatible sheet must contain Pair, Distribution, Covariate, "
            "ΔAIC/dAICc, beta, and LR p columns.\n\n"
            f"Workbooks searched:\n{searched or '- none'}"
        )

    matches.sort(
        key=lambda item: (
            item["score"],
            item["path"].stat().st_mtime,
        ),
        reverse=True,
    )
    return matches[0], matches


In [3]:
# ============================================================
# 3. LOAD AND STANDARDIZE THE SCREENING TABLE
# ============================================================

source, compatible_sources = find_compatible_screening_sheet(INPUT_XLSX)

SOURCE_XLSX = source["path"]
SOURCE_SHEET = source["sheet"]
COLUMN_MAPPING = source["mapping"]

raw_screen = pd.read_excel(
    SOURCE_XLSX,
    sheet_name=SOURCE_SHEET,
)

# Re-detect from the complete table in case the preview and full header differ.
COLUMN_MAPPING = detect_column_mapping(raw_screen.columns)

screen = pd.DataFrame(index=raw_screen.index)
screen["Pair"] = raw_screen[COLUMN_MAPPING["Pair"]].astype("string").str.strip()
screen["Pair label"] = screen["Pair"].map(pair_label)
screen["Distribution"] = (
    raw_screen[COLUMN_MAPPING["Distribution"]]
    .astype("string")
    .str.strip()
)
screen["Covariate raw"] = (
    raw_screen[COLUMN_MAPPING["Covariate"]]
    .astype("string")
    .str.strip()
)
screen["Covariate key"] = screen["Covariate raw"].map(canonical_covariate)
screen["Covariate"] = [
    covariate_label(key, raw)
    for key, raw in zip(screen["Covariate key"], screen["Covariate raw"])
]

for standard, output in [
    ("Delta_AIC", "Delta AIC"),
    ("Beta", "Beta"),
    ("LR_p", "LR p"),
]:
    screen[output] = pd.to_numeric(
        raw_screen[COLUMN_MAPPING[standard]],
        errors="coerce",
    )

if "Estimable" in COLUMN_MAPPING:
    screen["Estimable"] = to_nullable_boolean(
        raw_screen[COLUMN_MAPPING["Estimable"]]
    )
else:
    screen["Estimable"] = pd.Series(
        True,
        index=screen.index,
        dtype="boolean",
    )

if "Optimizer_converged" in COLUMN_MAPPING:
    screen["Optimizer converged"] = to_nullable_boolean(
        raw_screen[COLUMN_MAPPING["Optimizer_converged"]]
    )
else:
    screen["Optimizer converged"] = pd.Series(
        True,
        index=screen.index,
        dtype="boolean",
    )

if "GOF_preserved" in COLUMN_MAPPING:
    screen["GOF acceptance preserved"] = to_nullable_boolean(
        raw_screen[COLUMN_MAPPING["GOF_preserved"]]
    )
    GOF_AVAILABLE = screen["GOF acceptance preserved"].notna().any()
else:
    screen["GOF acceptance preserved"] = pd.Series(
        pd.NA,
        index=screen.index,
        dtype="boolean",
    )
    GOF_AVAILABLE = False

source_delta_column = COLUMN_MAPPING["Delta_AIC"]
METRIC_LABEL = (
    "ΔAICc"
    if "aicc" in normalized_token(source_delta_column)
    else "ΔAIC"
)

# Pair order follows the source table. Covariates use the requested order,
# followed by any additional covariates discovered in the workbook.
PAIR_ORDER = screen["Pair"].drop_duplicates().tolist()
PAIR_LABELS = (
    screen[["Pair", "Pair label"]]
    .drop_duplicates()
    .set_index("Pair")["Pair label"]
    .to_dict()
)

discovered_covariates = screen["Covariate key"].drop_duplicates().tolist()
COVARIATE_KEYS = [
    key for key in COVARIATE_ORDER
    if key in discovered_covariates
] + [
    key for key in discovered_covariates
    if key not in COVARIATE_ORDER
]

COVARIATE_LABELS = (
    screen[["Covariate key", "Covariate"]]
    .drop_duplicates("Covariate key")
    .set_index("Covariate key")["Covariate"]
    .to_dict()
)

print(f"Input workbook : {SOURCE_XLSX.name}")
print(f"Results sheet  : {SOURCE_SHEET}")
print(f"Rows read      : {len(screen):,}")
print(f"Pairs          : {screen['Pair'].nunique()}")
print(f"Distributions  : {screen['Distribution'].nunique()}")
print(f"Covariates     : {screen['Covariate key'].nunique()}")
print(f"Metric         : {METRIC_LABEL}")
print(
    "GOF criterion  : "
    + (
        "available and will be applied"
        if GOF_AVAILABLE and REQUIRE_GOF_PRESERVED_IF_AVAILABLE
        else "available but disabled"
        if GOF_AVAILABLE
        else "not available in the source workbook"
    )
)


Input workbook : T4_41_Distribution_Covariate_Screening.xlsx
Results sheet  : S10_Full_screen
Rows read      : 287
Pairs          : 8
Distributions  : 6
Covariates     : 7
Metric         : ΔAIC
GOF criterion  : available and will be applied


## Meaningful-improvement rule

The likelihood-ratio p-value answers whether adding the covariate improves the
fit beyond the baseline distribution. The information-criterion change measures
the strength of that improvement:

\[
\Delta\mathrm{AIC}
=\mathrm{AIC}_{\mathrm{baseline}}
-\mathrm{AIC}_{\mathrm{covariate}}.
\]

Therefore, a **larger positive value is better**. A blank output cell means that
no tested distribution for that pair–covariate combination passed every active
criterion.


In [4]:
# ============================================================
# 4. APPLY THE RULE AND FIND EACH PAIR × COVARIATE WINNER
# ============================================================

def exclusion_reasons(row):
    reasons = []

    if pd.isna(row["Estimable"]) or not bool(row["Estimable"]):
        reasons.append("not estimable")

    if (
        pd.isna(row["Optimizer converged"])
        or not bool(row["Optimizer converged"])
    ):
        reasons.append("optimizer did not converge")

    if pd.isna(row["Delta AIC"]):
        reasons.append(f"missing {METRIC_LABEL}")
    elif row["Delta AIC"] < MIN_DELTA_AIC:
        reasons.append(f"{METRIC_LABEL} < {MIN_DELTA_AIC:g}")

    if pd.isna(row["LR p"]):
        reasons.append("missing LR p")
    elif row["LR p"] >= ALPHA_LR:
        reasons.append(f"LR p ≥ {ALPHA_LR:g}")

    if pd.isna(row["Beta"]):
        reasons.append("missing β")

    if GOF_AVAILABLE and REQUIRE_GOF_PRESERVED_IF_AVAILABLE:
        if (
            pd.isna(row["GOF acceptance preserved"])
            or not bool(row["GOF acceptance preserved"])
        ):
            reasons.append("GOF acceptance not preserved")

    return "; ".join(reasons) if reasons else "Qualifies"


screen["Selection status"] = screen.apply(exclusion_reasons, axis=1)
screen["Qualifies"] = screen["Selection status"].eq("Qualifies")
screen["Effect ratio exp(β)"] = np.exp(screen["Beta"])
screen["Effect direction"] = np.select(
    [
        screen["Beta"] > 0,
        screen["Beta"] < 0,
    ],
    [
        "Longer headway",
        "Shorter headway",
    ],
    default="No directional change",
)

qualifying = screen.loc[screen["Qualifies"]].copy()

# Stable ordering and deterministic tie-breaking:
# 1. largest information-criterion improvement;
# 2. smallest LR p;
# 3. largest absolute beta;
# 4. alphabetical distribution name.
pair_rank = {pair: rank for rank, pair in enumerate(PAIR_ORDER)}
covariate_rank = {
    covariate: rank
    for rank, covariate in enumerate(COVARIATE_KEYS)
}

qualifying["_pair rank"] = qualifying["Pair"].map(pair_rank)
qualifying["_covariate rank"] = qualifying["Covariate key"].map(covariate_rank)
qualifying["_absolute beta"] = qualifying["Beta"].abs()

qualifying = qualifying.sort_values(
    [
        "_pair rank",
        "_covariate rank",
        "Delta AIC",
        "LR p",
        "_absolute beta",
        "Distribution",
    ],
    ascending=[True, True, False, True, False, True],
)

winners = (
    qualifying
    .drop_duplicates(
        subset=["Pair", "Covariate key"],
        keep="first",
    )
    .copy()
)


# Requested maximum-improvement matrix
maximum_delta_matrix = (
    winners
    .pivot(
        index="Pair",
        columns="Covariate key",
        values="Delta AIC",
    )
    .reindex(
        index=PAIR_ORDER,
        columns=COVARIATE_KEYS,
    )
    .rename(
        index=PAIR_LABELS,
        columns=COVARIATE_LABELS,
    )
)
maximum_delta_matrix.index.name = "Pair"


# Distribution that produces each matrix value
winning_distribution_matrix = (
    winners
    .pivot(
        index="Pair",
        columns="Covariate key",
        values="Distribution",
    )
    .reindex(
        index=PAIR_ORDER,
        columns=COVARIATE_KEYS,
    )
    .rename(
        index=PAIR_LABELS,
        columns=COVARIATE_LABELS,
    )
)
winning_distribution_matrix.index.name = "Pair"


# Combined presentation matrix: value plus winning distribution
combined_matrix = maximum_delta_matrix.copy().astype("object")

for pair in combined_matrix.index:
    for covariate in combined_matrix.columns:
        value = maximum_delta_matrix.loc[pair, covariate]
        distribution = winning_distribution_matrix.loc[pair, covariate]

        combined_matrix.loc[pair, covariate] = (
            f"{value:.3f} ({distribution})"
            if pd.notna(value)
            else "—"
        )

combined_matrix.index.name = "Pair"


# Strongest covariate overall for each pair
pair_best_rows = []

for pair in maximum_delta_matrix.index:
    available = maximum_delta_matrix.loc[pair].dropna()

    if available.empty:
        pair_best_rows.append(
            {
                "Pair": pair,
                "Strongest covariate": "None",
                f"Maximum {METRIC_LABEL}": np.nan,
                "Winning distribution": "None",
            }
        )
        continue

    strongest_covariate = available.idxmax()
    pair_best_rows.append(
        {
            "Pair": pair,
            "Strongest covariate": strongest_covariate,
            f"Maximum {METRIC_LABEL}": available.loc[strongest_covariate],
            "Winning distribution": winning_distribution_matrix.loc[
                pair,
                strongest_covariate,
            ],
        }
    )

pair_best_covariate = pd.DataFrame(pair_best_rows)


# Long-form winning model details
winner_details = winners[
    [
        "Pair label",
        "Covariate",
        "Distribution",
        "Delta AIC",
        "LR p",
        "Beta",
        "Effect ratio exp(β)",
        "Effect direction",
        "GOF acceptance preserved",
    ]
].copy()

winner_details = winner_details.rename(
    columns={
        "Pair label": "Pair",
        "Delta AIC": METRIC_LABEL,
        "Beta": "β",
    }
).sort_values(
    ["Pair", "Covariate"],
    ignore_index=True,
)


# Full selection audit
selection_audit = screen[
    [
        "Pair label",
        "Distribution",
        "Covariate",
        "Delta AIC",
        "LR p",
        "Beta",
        "Effect ratio exp(β)",
        "Effect direction",
        "Estimable",
        "Optimizer converged",
        "GOF acceptance preserved",
        "Qualifies",
        "Selection status",
    ]
].copy()

selection_audit = selection_audit.rename(
    columns={
        "Pair label": "Pair",
        "Delta AIC": METRIC_LABEL,
        "Beta": "β",
    }
)


print(f"Qualifying fitted models: {len(qualifying):,} of {len(screen):,}")
print(f"Pair × covariate winners: {len(winners):,}")


Qualifying fitted models: 61 of 287
Pair × covariate winners: 18


In [5]:
# ============================================================
# 5. DISPLAY THE REQUESTED TABLES
# ============================================================

print(
    f"Maximum meaningful {METRIC_LABEL} improvement by pair and covariate\n"
    f"Active rule: {METRIC_LABEL} ≥ {MIN_DELTA_AIC:g}, "
    f"LR p < {ALPHA_LR:g}"
    + (
        ", GOF acceptance preserved"
        if GOF_AVAILABLE and REQUIRE_GOF_PRESERVED_IF_AVAILABLE
        else ""
    )
)

if HAS_IPYTHON_DISPLAY:
    display(
        maximum_delta_matrix.style
        .format("{:.3f}", na_rep="—")
        .background_gradient(
            cmap="YlGn",
            axis=None,
            subset=maximum_delta_matrix.columns,
        )
        .highlight_max(
            axis=1,
            color="#A9D18E",
        )
    )
else:
    display(maximum_delta_matrix.round(3))

print("\nDistribution producing each maximum value")
display(winning_distribution_matrix.fillna("—"))

print("\nStrongest covariate for each pair")
if HAS_IPYTHON_DISPLAY:
    display(
        pair_best_covariate.style.format(
            {f"Maximum {METRIC_LABEL}": "{:.3f}"},
            na_rep="—",
        )
    )
else:
    display(pair_best_covariate.round({f"Maximum {METRIC_LABEL}": 3}))

print("\nWinning-model details")
if HAS_IPYTHON_DISPLAY:
    display(
        winner_details.style.format(
            {
                METRIC_LABEL: "{:.3f}",
                "LR p": "{:.4f}",
                "β": "{:.4f}",
                "Effect ratio exp(β)": "{:.4f}",
            },
            na_rep="—",
        )
    )
else:
    display(
        winner_details.round(
            {
                METRIC_LABEL: 3,
                "LR p": 4,
                "β": 4,
                "Effect ratio exp(β)": 4,
            }
        )
    )


Maximum meaningful ΔAIC improvement by pair and covariate
Active rule: ΔAIC ≥ 2, LR p < 0.05, GOF acceptance preserved


Covariate key,Target Vehicle Speed,Leading Vehicle Speed,Speed Difference,Occupancy,Off-centeredness,Site,Flow
Pair,,,,,,,
PR → 4W,—,—,—,—,3.120,—,—
PR → MT_3W,17.179,—,—,7.369,—,—,—
PR → NMT_3W,—,—,—,6.695,—,6.733,—
BTW → 4W,27.368,—,30.023,—,4.220,4.278,—
BTW → MT_3W,41.905,—,40.574,—,3.531,12.744,—
BTW → MT_2W,4.857,—,5.723,—,—,—,—
BTW → NMT_3W,4.113,5.833,—,—,—,—,—
BTW → NMT_2W,—,—,7.685,—,—,—,—



Distribution producing each maximum value


Covariate key,Target Vehicle Speed,Leading Vehicle Speed,Speed Difference,Occupancy,Off-centeredness,Site,Flow
Pair,,,,,,,
PR → 4W,—,—,—,—,Inverse Gaussian,—,—
PR → MT_3W,Pearson type III,—,—,Gamma,—,—,—
PR → NMT_3W,—,—,—,Generalized gamma,—,Generalized gamma,—
BTW → 4W,Pearson type III,—,Pearson type III,—,Gamma,Weibull,—
BTW → MT_3W,Log-normal,—,Inverse Gaussian,—,Inverse Gaussian,Gamma,—
BTW → MT_2W,Weibull,—,Gamma,—,—,—,—
BTW → NMT_3W,Weibull,Weibull,—,—,—,—,—
BTW → NMT_2W,—,—,Inverse Gaussian,—,—,—,—



Strongest covariate for each pair


,Pair,Strongest covariate,Maximum ΔAIC,Winning distribution
0,PR → 4W,Off-centeredness,3.120,Inverse Gaussian
1,PR → MT_3W,Target Vehicle Speed,17.179,Pearson type III
2,PR → NMT_3W,Site,6.733,Generalized gamma
3,BTW → 4W,Speed Difference,30.023,Pearson type III
4,BTW → MT_3W,Target Vehicle Speed,41.905,Log-normal
5,BTW → MT_2W,Speed Difference,5.723,Gamma
6,BTW → NMT_3W,Leading Vehicle Speed,5.833,Weibull
7,BTW → NMT_2W,Speed Difference,7.685,Inverse Gaussian



Winning-model details


,Pair,Covariate,Distribution,ΔAIC,LR p,β,Effect ratio exp(β),Effect direction,GOF acceptance preserved
0,BTW → 4W,Off-centeredness,Gamma,4.220,0.0126,-0.1395,0.8698,Shorter headway,True
1,BTW → 4W,Site,Weibull,4.278,0.0122,0.1304,1.1393,Longer headway,True
2,BTW → 4W,Speed Difference,Pearson type III,30.023,0.0000,-0.1305,0.8776,Shorter headway,True
3,BTW → 4W,Target Vehicle Speed,Pearson type III,27.368,0.0000,-0.1189,0.8879,Shorter headway,True
4,BTW → MT_2W,Speed Difference,Gamma,5.723,0.0055,-0.1397,0.8696,Shorter headway,True
5,BTW → MT_2W,Target Vehicle Speed,Weibull,4.857,0.0088,-0.1374,0.8716,Shorter headway,True
6,BTW → MT_3W,Off-centeredness,Inverse Gaussian,3.531,0.0187,-0.1680,0.8454,Shorter headway,True
7,BTW → MT_3W,Site,Gamma,12.744,0.0001,0.2595,1.2963,Longer headway,True
8,BTW → MT_3W,Speed Difference,Inverse Gaussian,40.574,0.0000,-0.2122,0.8088,Shorter headway,True
9,BTW → MT_3W,Target Vehicle Speed,Log-normal,41.905,0.0000,-0.2164,0.8054,Shorter headway,True


In [6]:
# ============================================================
# 6. EXPORT A FORMATTED EXCEL SUMMARY
# ============================================================

from openpyxl import load_workbook
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


criteria = pd.DataFrame(
    {
        "Item": [
            "Source workbook",
            "Source sheet",
            "Source rows",
            "Models represented",
            "Pairs represented",
            "Covariates represented",
            "Information criterion",
            "Minimum improvement",
            "Significance test",
            "Significance threshold",
            "GOF preservation",
            "Cell definition",
            "Blank-cell meaning",
            "Beta interpretation",
        ],
        "Specification": [
            SOURCE_XLSX.name,
            SOURCE_SHEET,
            len(screen),
            screen[["Pair", "Distribution"]].drop_duplicates().shape[0],
            screen["Pair"].nunique(),
            screen["Covariate key"].nunique(),
            METRIC_LABEL,
            f"{METRIC_LABEL} ≥ {MIN_DELTA_AIC:g}",
            "Likelihood-ratio test",
            f"LR p < {ALPHA_LR:g}",
            (
                "Required"
                if GOF_AVAILABLE and REQUIRE_GOF_PRESERVED_IF_AVAILABLE
                else "Available but not required"
                if GOF_AVAILABLE
                else "Not available in source workbook; not applied"
            ),
            (
                f"Maximum qualifying {METRIC_LABEL} across distributions "
                "for each pair × covariate"
            ),
            "No tested distribution passed every active criterion",
            (
                "β > 0 indicates longer headway; β < 0 indicates shorter "
                "headway under the fitted AFT scale model"
            ),
        ],
    }
)


OUTPUT_XLSX = Path(OUTPUT_XLSX)
OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    criteria.to_excel(
        writer,
        sheet_name="S0_Criteria",
        index=False,
    )

    maximum_delta_matrix.reset_index().to_excel(
        writer,
        sheet_name="S1_Maximum_dAIC",
        index=False,
    )

    winning_distribution_matrix.reset_index().to_excel(
        writer,
        sheet_name="S2_Winning_distribution",
        index=False,
    )

    combined_matrix.reset_index().to_excel(
        writer,
        sheet_name="S3_Combined",
        index=False,
    )

    pair_best_covariate.to_excel(
        writer,
        sheet_name="S4_Pair_best_covariate",
        index=False,
    )

    winner_details.to_excel(
        writer,
        sheet_name="S5_Winner_details",
        index=False,
    )

    qualifying.drop(
        columns=[
            "_pair rank",
            "_covariate rank",
            "_absolute beta",
        ],
        errors="ignore",
    ).to_excel(
        writer,
        sheet_name="S6_All_qualifying",
        index=False,
    )

    selection_audit.to_excel(
        writer,
        sheet_name="S7_Selection_audit",
        index=False,
    )


def style_summary_workbook(path):
    workbook = load_workbook(path)

    navy = "17365D"
    teal = "0F6B78"
    light_blue = "D9EAF7"
    light_green = "E2F0D9"
    row_max_green = "A9D18E"
    white = "FFFFFF"
    gray = "666666"
    thin_gray = Side(style="thin", color="D9E2F3")

    tab_colors = [
        "5B9BD5",
        "70AD47",
        "A5A5A5",
        "ED7D31",
        "4472C4",
        "264478",
        "255E91",
        "636363",
    ]

    for sheet_number, worksheet in enumerate(workbook.worksheets):
        worksheet.sheet_view.showGridLines = False
        worksheet.freeze_panes = "A2"
        worksheet.sheet_properties.tabColor = tab_colors[
            sheet_number % len(tab_colors)
        ]

        if worksheet.max_row >= 1 and worksheet.max_column >= 1:
            worksheet.auto_filter.ref = worksheet.dimensions

        for cell in worksheet[1]:
            cell.fill = PatternFill("solid", fgColor=navy)
            cell.font = Font(color=white, bold=True)
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )
            cell.border = Border(bottom=thin_gray)

        worksheet.row_dimensions[1].height = 32

        for column_index in range(1, worksheet.max_column + 1):
            column_letter = get_column_letter(column_index)
            header = worksheet.cell(1, column_index).value

            observed_width = max(
                (
                    len(str(worksheet.cell(row_index, column_index).value))
                    if worksheet.cell(row_index, column_index).value is not None
                    else 0
                )
                for row_index in range(
                    1,
                    min(worksheet.max_row, 250) + 1,
                )
            )

            if column_index == 1:
                width = min(max(observed_width + 2, 18), 30)
            else:
                width = min(max(observed_width + 2, 12), 34)

            worksheet.column_dimensions[column_letter].width = width

            header_text = str(header) if header is not None else ""
            for row_index in range(2, worksheet.max_row + 1):
                cell = worksheet.cell(row_index, column_index)
                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=(
                        worksheet.title in {
                            "S0_Criteria",
                            "S3_Combined",
                            "S7_Selection_audit",
                        }
                    ),
                )

                if any(
                    marker in header_text
                    for marker in [METRIC_LABEL, "LR p", "β", "exp(β)"]
                ) and isinstance(cell.value, (int, float)):
                    cell.number_format = "0.0000"

        if worksheet.title == "S0_Criteria":
            worksheet.column_dimensions["A"].width = 30
            worksheet.column_dimensions["B"].width = 70
            for row_index in range(2, worksheet.max_row + 1):
                worksheet.cell(row_index, 1).font = Font(
                    bold=True,
                    color=teal,
                )
                if row_index % 2 == 0:
                    for column_index in range(1, 3):
                        worksheet.cell(
                            row_index,
                            column_index,
                        ).fill = PatternFill("solid", fgColor=light_blue)

    # Add a heat scale and explicitly mark the maximum covariate per pair.
    matrix_sheet = workbook["S1_Maximum_dAIC"]
    if matrix_sheet.max_row >= 2 and matrix_sheet.max_column >= 2:
        for row_index in range(2, matrix_sheet.max_row + 1):
            for column_index in range(2, matrix_sheet.max_column + 1):
                matrix_sheet.cell(
                    row_index,
                    column_index,
                ).number_format = "0.000"

        matrix_range = (
            f"B2:{get_column_letter(matrix_sheet.max_column)}"
            f"{matrix_sheet.max_row}"
        )

        matrix_sheet.conditional_formatting.add(
            matrix_range,
            ColorScaleRule(
                start_type="min",
                start_color="FFF2CC",
                mid_type="percentile",
                mid_value=50,
                mid_color="C6E0B4",
                end_type="max",
                end_color="548235",
            ),
        )

        for row_index in range(2, matrix_sheet.max_row + 1):
            numeric_cells = [
                matrix_sheet.cell(row_index, column_index)
                for column_index in range(2, matrix_sheet.max_column + 1)
                if isinstance(
                    matrix_sheet.cell(row_index, column_index).value,
                    (int, float),
                )
            ]
            if not numeric_cells:
                continue

            row_maximum = max(cell.value for cell in numeric_cells)
            for cell in numeric_cells:
                if np.isclose(cell.value, row_maximum):
                    cell.fill = PatternFill(
                        "solid",
                        fgColor=row_max_green,
                    )
                    cell.font = Font(bold=True, color="1F1F1F")

    # Shade qualifying rows in the audit for quick inspection.
    audit_sheet = workbook["S7_Selection_audit"]
    headers = {
        cell.value: cell.column
        for cell in audit_sheet[1]
    }
    qualifies_column = headers.get("Qualifies")
    if qualifies_column is not None:
        for row_index in range(2, audit_sheet.max_row + 1):
            if audit_sheet.cell(row_index, qualifies_column).value is True:
                for column_index in range(1, audit_sheet.max_column + 1):
                    audit_sheet.cell(
                        row_index,
                        column_index,
                    ).fill = PatternFill("solid", fgColor=light_green)

    workbook.properties.title = "Maximum covariate effect by pair"
    workbook.properties.subject = (
        "Maximum meaningful information-criterion improvement "
        "across fitted headway distributions"
    )
    workbook.properties.creator = "Tashfia Hassan"
    workbook.save(path)


style_summary_workbook(OUTPUT_XLSX)

print(f"Saved summary workbook: {OUTPUT_XLSX.resolve()}")


Saved summary workbook: D:\Headway\1. Codes\T4_Maximum_Covariate_Effect_by_Pair.xlsx


## Reading the result

- `S1_Maximum_dAIC` is the requested pair × covariate matrix.
- `S2_Winning_distribution` identifies the distribution responsible for each
  matrix value.
- `S3_Combined` puts the value and distribution in one cell.
- `S4_Pair_best_covariate` identifies the single strongest covariate for every
  pair.
- `S5_Winner_details` reports \(\beta\), \(\exp(\beta)\), LR p-value, effect
  direction, and GOF preservation for each winner.
- `S7_Selection_audit` shows why every tested model qualified or failed.

For continuous covariates standardized within a pair, \(\exp(\beta)\) is the
headway-time ratio for a one-standard-deviation increase. For binary variables,
it compares category 1 with category 0 under the coding used in the screening
notebook.
